# fes-bench：table[translated]

[translated] `fes-bench` [translated] `result/tables/*.csv` [translated] `data/processed/`，[translated] 2a–6 [translated]，export[translated] Excel [translated] Markdown[translated]

**[translated] notebook [translated]，[translated]** [translated]skill score（[translated] ΔG MAE [translated]phase[translated]）[translated]，[translated]

| [translated] | source CSV |
|---|---|
| [translated] 2a system[translated] | `system_inventory.csv` |
| [translated] 2b phase[translated] | `phase_inventory.csv` |
| [translated] 3 split | `split_definitions.csv` |
| [translated] 4 [translated] | `metric_definitions.csv` |
| [translated] 5a / 5b / 5c [translated] | `predictor_comparison_{temp_extrap,phase_lopo,system_loso}.csv` |
| [translated] 6 [translated]phase $T_c$ [translated] | `crossing_errors_temp_extrap.csv` |

**[translated]**：[translated] `SOURCE`，[translated] Runtime → Run all[translated] `fes_bench_outputs.zip`[translated]

In [ ]:
# ===== [translated] =====
SOURCE = "github"          # "github" [translated] | "upload" [translated] zip | "local" [translated]
REPO_URL = "https://github.com/zirenjin/fes-bench.git"
BRANCH = ""                # [translated] = [translated]
LOCAL_ROOT = "fes-bench"   # [translated]（local [translated]）
OUT_DIR = "fes_bench_outputs"

INCLUDE_PLACEHOLDER_ROWS = True   # [translated]row[translated] predictor [translated]row（TIP[translated]E+F_QH [translated]）
SHOW_FIXTURE_ROWS = False         # [translated] reference / [translated] / [translated]row，[translated]

In [ ]:
# ===== [translated]data =====
import os, subprocess, zipfile, pathlib, shutil, hashlib, json

def _run(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

root = pathlib.Path(LOCAL_ROOT)
if SOURCE == "github":
    if (root / ".git").exists():
        _run(["git", "-C", str(root), "pull", "-q"])
    else:
        cmd = ["git", "clone", "-q", "--depth", "1"] + (["-b", BRANCH] if BRANCH else []) + [REPO_URL, str(root)]
        _run(cmd)
elif SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()          # [translated] zip（[translated] result/tables [translated] data/processed）
    name = next(iter(uploaded))
    shutil.rmtree(root, ignore_errors=True); root.mkdir(parents=True)
    with zipfile.ZipFile(name) as z: z.extractall(root)
    hits = list(root.rglob("result/tables"))
    if not hits: raise FileNotFoundError("zip [translated] result/tables/")
    root = hits[0].parent.parent       # zip [translated]
elif SOURCE != "local":
    raise ValueError(SOURCE)

TABLES = root / "result" / "tables"
DATA = root / "data" / "processed"
try:
    COMMIT = subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    COMMIT = "unknown"
print("repo root:", root.resolve(), "| commit:", COMMIT)
print("tables:", sorted(p.name for p in TABLES.glob("*.csv")))

In [ ]:
# ===== [translated] =====
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 80)

def read_csv(name):
    # all[translated]，[translated] n/a:* [translated]empty[translated]
    return pd.read_csv(TABLES / f"{name}.csv", dtype=str, keep_default_na=False)

def read_meta(name):
    p = TABLES / f"{name}.meta.json"
    return json.loads(p.read_text()) if p.exists() else {}

RAW = {p.stem: read_csv(p.stem) for p in sorted(TABLES.glob("*.csv"))}
META = {k: read_meta(k) for k in RAW}
SPLITS = ["temp_extrap", "phase_lopo", "system_loso"]
SPLIT_LABEL = {"temp_extrap": "[translated]5a [translated]", "phase_lopo": "[translated]5b [translated]phase", "system_loso": "[translated]5c [translated]system"}

NA_LABEL = {
    "n/a:undefined_crossing_slope": "n/a:undefined_crossing_slope",
    "n/a:no_seed": "n/a:no_seed",
    "n/a:input_unavailable": "n/a:input_unavailable",
    "n/a:not_trained_for_split": "n/a:not_trained_for_split",
    "n/a:degenerate_prediction": "n/a:degenerate_prediction",
    "n/a:missed_crossing": "n/a:missed_crossing",
    "n/a:no_reference_crossing": "n/a（no reference crossing）",
    "n/a:pair_only_predictor":   "n/a（pair-only ΔG predictor）",
    "n/a:no_training_phase":     "n/a（no training phase）",
}
MISSING = "—"   # empty = [translated]，[translated]

def num(x):
    try: return float(x)
    except (TypeError, ValueError): return np.nan

def is_marker(x): return isinstance(x, str) and x.startswith("n/a:")

def fmt(x, scale=1.0, digits=2, signed=False):
    # [translated]；n/a [translated]；empty[translated] —
    if is_marker(x): return NA_LABEL.get(x, x)
    v = num(x)
    if np.isnan(v): return MISSING
    v *= scale
    return f"{v:+.{digits}f}" if signed else f"{v:.{digits}f}"

def sha256(p):
    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

# predictor [translated] predictor [translated]"[translated]"[translated]，[translated]
PREDICTORS = {
    "zero":                ("zero",                          "[translated]", 10),
    "constant_delta_g":    ("constant ΔG",                   "[translated]", 11),
    "global_mean_delta_g": ("global-mean ΔG",                "[translated]", 12),
    "interp_const":        ("interp_constant",               "[translated]", 13),
    "phase_id_mlp":        ("phase_id_mlp",                  "[translated]", 14),
    "bartel2018":          ("Bartel descriptor",             "[translated]", 20),
    "tip_uma":             ("TIP[UMA]†",                     "[translated]", 21),
    "qh_only":             ("E + F_QH",                      "[translated]", 30),
    "qh_residual":         ("E + F_QH + r_θ",                "[translated]", 31),
    "repr_only":           ("representation regression",     "[translated]", 32),
    "polynomial":          ("repr. regression (polynomial)", "[translated]", 33),
    "tlog_polynomial":     ("repr. regression (tlog)",       "[translated]", 34),
}
PLACEHOLDERS = ["tip_uma", "qh_only", "qh_residual", "repr_only"]
FIXTURE_PREFIXES = ("reference", "root_stability", "constant_offset")
GROUP_ORDER = ["[translated]", "[translated]", "[translated]", "[translated]", "[translated]"]

def pinfo(key):
    if key.startswith(FIXTURE_PREFIXES): return (key, "[translated]", 90)
    return PREDICTORS.get(key, (key, "[translated]", 80))

print({k: v.shape for k, v in RAW.items()})

## [translated]check

**[translated]** [translated]check CSV [translated] ⚠ [translated]，[translated]table[translated]，[translated]

In [ ]:
# ===== [translated]check =====
CHECKS = []
def check(ok, msg, level="WARN"):
    tag = "✓" if ok else ("⚠" if level == "WARN" else "ℹ")
    CHECKS.append(f"{tag} {msg}"); print(CHECKS[-1])

pc = {s: RAW.get(f"predictor_comparison_{s}") for s in SPLITS}
metric_cols = ["delta_G_MAE_eV_per_atom", "sign_accuracy", "Tc_error_K"]

# 1. [translated]split[translated]
for p in sorted(set().union(*[set(df.predictor) for df in pc.values() if df is not None])):
    rows = [tuple(df.loc[df.predictor == p, metric_cols].iloc[0]) for df in pc.values()
            if df is not None and (df.predictor == p).any()]
    vals = [r for r in rows if any(num(x) == num(x) for x in r)]
    if len(vals) >= 2 and len(set(vals)) == 1 and not p.startswith(FIXTURE_PREFIXES):
        check(False, f"{p}: [translated]split[translated] ΔG MAE / [translated] / Tc [translated]phase[translated] —— [translated]split[translated]")

# 2. LOPO/LOSO [translated] constant ΔG [translated]，[translated]
for s in ["phase_lopo", "system_loso"]:
    df = pc[s]
    if df is not None and (df.predictor == "constant_delta_g").any():
        r = df[df.predictor == "constant_delta_g"].iloc[0]
        z = df[df.predictor == "zero"].iloc[0] if (df.predictor == "zero").any() else None
        has = not np.isnan(num(r.delta_G_MAE_eV_per_atom))
        same_as_zero = z is not None and tuple(r[metric_cols]) == tuple(z[metric_cols])
        check(not has, f"{s}: constant ΔG [translated]（[translated] n/a:no_training_phase）"
              + ("，[translated] zero row[translated]phase[translated]" if same_as_zero else ""))

# 3. zero [translated] ΔG≡0，crossing[translated]
for s, df in pc.items():
    if df is not None and (df.predictor == "zero").any():
        v = num(df[df.predictor == "zero"].iloc[0].Tc_error_K)
        check(np.isnan(v), f"{s}: zero row Tc [translated] = {v:.1f} K（ΔG≡0 [translated]，[translated] n/a）")

# 4. [translated]column
need = {"ranking_accuracy": "phase-order accuracy", "skill_score": "skill score", "Tc_err_from_dG_K": "[translated]",
        "seed_std": "seed [translated]"}
for s, df in pc.items():
    if df is not None:
        miss = [v for k, v in need.items() if k not in df.columns]
        check(not miss, f"{s}: [translated] CSV [translated]column {miss}（notebook [translated]skill score[translated]，[translated] —）", "INFO")

# 5. [translated]row[translated]
present = set().union(*[set(df.predictor) for df in pc.values() if df is not None])
check(bool(present & {"qh_only", "qh_residual", "repr_only", "polynomial", "tlog_polynomial"}),
      "[translated]row（E1 [translated] polynomial / tlog check[translated]）", "INFO")

# 6. qh_reliable [translated]phase[translated]
si, ph = RAW.get("system_inventory"), RAW.get("phase_inventory")
if si is not None and ph is not None:
    for _, r in si.iterrows():
        sub = ph[ph.system == r.system]
        n_true = (sub.qh_reliable.str.lower() == "true").sum()
        n_blank = (sub.qh_reliable == "").sum()
        check(str(n_true) == r.qh_reliable_count and n_blank == 0,
              f"{r.system}: system_inventory [translated] qh_reliable_count={r.qh_reliable_count}，"
              f"phase_inventory [translated] true={n_true}[translated]={n_blank}")
    for _, r in si.iterrows():
        if "D3" in r.truth_level and "D3" not in r.functional:
            check(False, f"{r.system}: functional [translated] '{r.functional}'，[translated] '{r.truth_level}'")

# 7. temp_extrap [translated]crossing[translated]
sd = RAW.get("split_definitions")
if sd is not None:
    te = sd[sd.split == "temp_extrap"]
    bad = [x for c in te.crossing_in_test for x in c.split(";") if x and not x.endswith("=test")]
    check(not bad, f"temp_extrap crossingall[translated]" + (f"：[translated] {bad}" if bad else ""))

# 8. meta.json [translated]missing[translated]
for k, m in META.items():
    miss = m.get("missing", [])
    if miss:
        reasons = pd.Series([x.get("reason", "?") if isinstance(x, dict) else str(x) for x in miss]).value_counts()
        check(True, f"{k}: meta [translated] {len(miss)} [translated]missing → " + "; ".join(f"{i}×{n}" for i, n in reasons.items()), "INFO")

n_warn = sum(c.startswith("⚠") for c in CHECKS)
print(f"\n[translated] {n_warn} [translated]")

## [translated] 2a system[translated]

In [ ]:
def table_2a():
    df = RAW["system_inventory"]
    out = pd.DataFrame({
        "system": df.system,
        "phase": df.phases.str.replace(";", ", "),
        "phase[translated]": df.n_phases,
        "[translated]": df.truth_level,
        "[translated]": df.functional,
        "T [translated] (K)": [f"{fmt(a,digits=0)}–{fmt(b,digits=0)}" for a, b in zip(df.T_min_K, df.T_max_K)],
        "[translated]": df.grid_points,
        "ΔG std (meV/atom)": [fmt(a, digits=2) if a == b else f"{fmt(a,digits=2)}–{fmt(b,digits=2)}"
                               for a, b in zip(df.delta_G_std_min_meV_per_atom, df.delta_G_std_max_meV_per_atom)],
        "phase[translated]": df.relative_numbers,
        "QH [translated]": [f"{c}/{n}" for c, n in zip(df.qh_reliable_count, df.n_phases)],
        "DOI": df.doi,
    })
    return out
T2A = table_2a(); T2A

## [translated] 2b phase[translated]

In [ ]:
def table_2b():
    df = RAW["phase_inventory"]
    src = df.representative_source.map(lambda s: pathlib.PurePath(s).name if s else MISSING)
    return pd.DataFrame({
        "system": df.system, "phase": df.phase,
        "[translated]（[translated]→[translated]）": [f"{a or '?'} → {b or '?'}" for a, b in zip(df.space_group_before, df.space_group_after)],
        "[translated]": df.primitive_atoms,
        "[translated]source": src,
        "QH [translated]": df.qh_supercell, "[translated] (Å)": df.shortest_edge_A.map(lambda x: fmt(x, digits=2)),
        "q [translated]": df.q_mesh,
        "[translated]": df.imaginary_fraction.map(lambda x: fmt(x, scale=100, digits=2) + ("%" if num(x) == num(x) else "")),
        "qh_reliable": df.qh_reliable.replace("", MISSING),
    })
T2B = table_2b(); T2B

## [translated] 3 split

In [ ]:
def table_3():
    df = RAW["split_definitions"]
    def crossings(s):
        parts = [x for x in s.split(";") if x]
        return f"{sum(p.endswith('=test') for p in parts)}/{len(parts)} [translated]"
    return pd.DataFrame({
        "split": df.split, "[translated]": df.fold, "[translated]": df.held_out_unit,
        "[translated]": df.train_frames, "[translated]": df.test_frames,
        "[translated]crossing": df.crossing_in_test.map(crossings),
        "SHA-256": df.sha256.str[:10], "commit": df.commit.str[:7],
    })
T3 = table_3(); T3

## [translated] 4 [translated]

In [ ]:
def table_4():
    df = RAW["metric_definitions"].copy()
    df["source"] = "[translated]"
    extra = pd.DataFrame([
        ("skill_score", "1 − MAE_model / MAE_floor；floor [translated] constant ΔG，[translated] global-mean ΔG", "—", "↑", "notebook [translated]（[translated]）"),
        ("ranking_accuracy", "allphase[translated]", "fraction", "↑", "[translated]"),
        ("Tc_err_from_dG", "crossing[translated] |ΔG [translated]| / |dΔG/dT|，[translated]temperature", "K", "↓", "[translated]（[translated] crossing [translated]）"),
    ], columns=["metric", "definition", "unit", "direction", "source"])
    df = pd.concat([df, extra[~extra.metric.isin(df.metric)]], ignore_index=True)
    return df.rename(columns={"metric": "[translated]", "definition": "[translated]", "unit": "[translated]", "direction": "[translated]"})
T4 = table_4(); T4

## [translated] 5a / 5b / 5c [translated]

row[translated] → [translated] → [translated]**skill score[translated]**：`1 − MAE/MAE_floor`，[translated]skill score[translated]column，[translated]split[translated] crossing [translated] predictor [translated]crossing[translated]mean[translated]

In [ ]:
def floor_for(df):
    for key in ["constant_delta_g", "global_mean_delta_g"]:
        if (df.predictor == key).any():
            v = num(df[df.predictor == key].iloc[0].delta_G_MAE_eV_per_atom)
            if not np.isnan(v) and v > 0: return key, v
    return None, np.nan

def slope_conversion(split):
    ce = RAW.get(f"crossing_errors_{split}")
    if ce is None: return {}
    ce = ce.assign(v=ce.Tc_err_from_dG_K.map(num))
    return ce.groupby("predictor").v.apply(lambda s: s.abs().mean() if s.notna().any() else np.nan).to_dict()

def fm_counts(split):
    ce = RAW.get(f"crossing_errors_{split}")
    if ce is None: return {}
    out = {}
    for p, g in ce.groupby("predictor"):
        f, m = g.false_crossings.map(num), g.missed_crossings.map(num)
        out[p] = MISSING if f.isna().all() else f"{int(np.nansum(f))}/{int(np.nansum(m))}"
    return out

def table_5(split):
    df = RAW[f"predictor_comparison_{split}"].copy()
    if not SHOW_FIXTURE_ROWS:
        df = df[~df.predictor.str.startswith(FIXTURE_PREFIXES)]
    df = df.reset_index(drop=True)          # [translated]row[translated]row
    if INCLUDE_PLACEHOLDER_ROWS:
        for p in PLACEHOLDERS:
            if p not in set(df.predictor):
                df.loc[len(df)] = {c: "" for c in df.columns} | {"predictor": p}
    slope, fm = slope_conversion(split), fm_counts(split)
    rows = []
    for _, r in df.iterrows():
        name, group, order = pinfo(r.predictor)
        mae = num(r.delta_G_MAE_eV_per_atom)
        skill = num(r.get("skill_score", ""))
        rows.append({
            "_order": order, "[translated]": group, "predictor": name,
            "ΔG MAE ↓ (meV/atom)": fmt(r.delta_G_MAE_eV_per_atom, scale=1000, digits=2),
            "skill score ↑": MISSING if np.isnan(skill) else f"{skill:+.2f}",
            "sign accuracy ↑ (%)": fmt(r.sign_accuracy, scale=100, digits=1),
            "phase-order accuracy ↑ (%)": fmt(r.get("ranking_accuracy", ""), scale=100, digits=1),
            "T_c [translated] ↓ (K)": fmt(r.Tc_error_K, digits=1),
            "[translated] ↓ (K)": MISSING if np.isnan(slope.get(r.predictor, np.nan)) else f"{slope[r.predictor]:.0f}",
            "[translated]/[translated]crossing ↓": fm.get(r.predictor, MISSING),
            "seed [translated]": fmt(r.get("seed_std", ""), digits=2),
            "_skill": skill,
        })
    out = pd.DataFrame(rows)
    out["_g"] = out["[translated]"].map(GROUP_ORDER.index)
    out = out.sort_values(["_g", "_order"]).drop(columns=["_g", "_order"]).reset_index(drop=True)
    note = "skill score and floor_predictor are read directly from the table CSV"
    return out, note

T5 = {}
for s in SPLITS:
    t, note = table_5(s); T5[s] = (t, note)
    print(f"\n{SPLIT_LABEL[s]} —— {note}")
    display(t.drop(columns=["_skill"]))


## [translated] 6 [translated]phase $T_c$ [translated]（temp_extrap）

In [ ]:
def table_6(split="temp_extrap"):
    ce = RAW[f"crossing_errors_{split}"].copy()
    def pick_ref(s):   # [translated]；[translated] no_reference_crossing [translated]
        nums = [x for x in s if not np.isnan(num(x))]
        if nums: return nums[0]
        return next((x for x in s if x == "n/a:no_reference_crossing"), next(iter(s), ""))
    ref = ce.groupby("pair").agg(ref=("reference_Tc_K", pick_ref),
                                 slope=("crossing_slope_eV_per_atom_per_K",
                                        lambda s: next((x for x in s if num(x) > 0), "")))
    def cell(r):
        if is_marker(r.reference_Tc_K): 
            f = num(r.false_crossings)
            return NA_LABEL.get(r.reference_Tc_K) + (f" +{int(f)}[translated]" if f > 0 else "")
        if num(r.missed_crossings) > 0: return "[translated]"
        v = fmt(r.Tc_error_K, digits=0, signed=True)
        f = num(r.false_crossings)
        return v + (f" +{int(f)}[translated]" if f > 0 else "")
    if not SHOW_FIXTURE_ROWS:   # [translated] T_c [translated]allrow（[translated] reference）[translated]，[translated]row
        ce = ce[~ce.predictor.str.startswith(FIXTURE_PREFIXES)]
    ce["cell"] = ce.apply(cell, axis=1)
    piv = ce.pivot_table(index="pair", columns="predictor", values="cell", aggfunc="first")
    order = sorted(piv.columns, key=lambda p: (GROUP_ORDER.index(pinfo(p)[1]), pinfo(p)[2]))
    piv = piv[order].rename(columns=lambda p: pinfo(p)[0])
    piv.insert(0, "crossing[translated] dΔG/dT (meV/atom/K)", ref.slope.map(lambda x: fmt(x, scale=1000, digits=4)))
    piv.insert(0, "[translated] T_c (K)", ref.ref.map(lambda x: fmt(x, digits=1)))
    piv.index = [p.replace("_minus_", " − ") for p in piv.index]
    return piv.reset_index().rename(columns={"index": "phase[translated]"})

T6 = table_6(); T6

## [translated]

- **[translated] A**：[translated] ΔG(T) [translated]crossing，[translated] temp_extrap [translated] meV/atom[translated]
- **[translated] B**：ΔG MAE vs sign accuracy（[translated]split[translated]）[translated] ΔG [translated]standard deviation——MAE [translated]，[translated]
- **[translated] C**：skill score，[translated]
- **[translated] D**：[translated] predictor [translated]crossing[translated] $T_c$ [translated]，x [translated]crossing[translated]

In [ ]:
import itertools, matplotlib.pyplot as plt
FIG = pathlib.Path(OUT_DIR) / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
import matplotlib.font_manager as fm
def _cjk():
    return next((x.name for x in fm.fontManager.ttflist
                 if any(k in x.name for k in ["Noto Sans CJK", "SimHei", "WenQuanYi", "PingFang", "Microsoft YaHei"])), None)
if _cjk() is None:   # Colab [translated]，[translated]
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for f in pathlib.Path("/usr/share/fonts").rglob("NotoSansCJK*.tt*"): fm.fontManager.addfont(str(f))
    except Exception as e:
        print("[translated]，[translated]：", e)
if _cjk(): plt.rcParams["font.sans-serif"] = [_cjk()] + plt.rcParams["font.sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

def save(fig, name):
    for ext in ("png", "pdf"): fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight")
    plt.show()

# ---- [translated] A：[translated] ΔG(T) ----
thr = {}
tsplit = DATA / "splits" / "temp_extrap.json"
if tsplit.exists(): thr = json.loads(tsplit.read_text()).get("threshold_T_K", {})
systems = [d.name for d in sorted(DATA.iterdir()) if (d / "system.json").exists()]
fig, axes = plt.subplots(1, len(systems), figsize=(3.4 * len(systems), 2.8), squeeze=False)
for ax, sysn in zip(axes[0], systems):
    phases = json.loads((DATA / sysn / "system.json").read_text())["phases"]
    G = {p: pd.read_csv(DATA / sysn / p / "reference_G.csv") for p in phases
         if (DATA / sysn / p / "reference_G.csv").exists()}
    for a, b in itertools.combinations(G, 2):
        m = G[a].merge(G[b], on="T_K", suffixes=("_a", "_b"))
        d = (m.G_eV_per_atom_a - m.G_eV_per_atom_b) * 1000
        ax.plot(m.T_K, d, lw=1.4, label=f"{a} − {b}")
        # [translated]；[translated]，[translated] 0 [translated]（[translated]）
        v = d.values; nz = np.flatnonzero(v != 0)
        for i, j in zip(nz[:-1], nz[1:]):
            if np.sign(v[i]) != np.sign(v[j]):
                t = m.T_K.iloc[i] - v[i] * (m.T_K.iloc[j] - m.T_K.iloc[i]) / (v[j] - v[i])
                ax.plot(t, 0, "ko", ms=4)
    if sysn in thr: ax.axvspan(ax.get_xlim()[0], thr[sysn], color="0.9", zorder=0)
    ax.axhline(0, color="0.4", lw=0.7); ax.set_title(sysn); ax.set_xlabel("T (K)")
    ax.legend(fontsize=6, frameon=False)
axes[0][0].set_ylabel("[translated] ΔG (meV/atom)")
fig.tight_layout(); save(fig, "A_reference_delta_g")

In [ ]:
# ---- [translated] B：MAE vs sign accuracy ----
ref_std = {}
si = RAW.get("system_inventory")
if si is not None:
    v = pd.to_numeric(si.delta_G_std_max_meV_per_atom, errors="coerce")
    ref_std = {"median": float(v.median())}
COL = {"[translated]": "tab:gray", "[translated]": "tab:orange", "[translated]": "tab:blue", "[translated]": "tab:purple"}
fig, axes = plt.subplots(1, len(SPLITS), figsize=(3.6 * len(SPLITS), 3.0), sharey=True)
for ax, s in zip(np.atleast_1d(axes), SPLITS):
    df = RAW[f"predictor_comparison_{s}"]
    df = df[~df.predictor.str.startswith(FIXTURE_PREFIXES)]
    for _, r in df.iterrows():
        x, y = num(r.delta_G_MAE_eV_per_atom) * 1000, num(r.sign_accuracy) * 100
        if np.isnan(x) or np.isnan(y): continue
        name, group, _ = pinfo(r.predictor)
        ax.scatter(x, y, color=COL.get(group, "k"), s=28)
        ax.annotate(name, (x, y), fontsize=6, xytext=(3, 3), textcoords="offset points")
    if ref_std: ax.axvline(ref_std["median"], ls="--", color="0.5", lw=0.8)
    ax.set_title(SPLIT_LABEL[s]); ax.set_xlabel("ΔG MAE (meV/atom)")
np.atleast_1d(axes)[0].set_ylabel("sign accuracy (%)")
fig.tight_layout(); save(fig, "B_mae_vs_sign")

# ---- [translated] C：skill score ----
fig, axes = plt.subplots(1, len(SPLITS), figsize=(3.8 * len(SPLITS), 3.0))
for ax, s in zip(np.atleast_1d(axes), SPLITS):
    t, note = T5[s]; t = t[t._skill.notna()]
    ax.barh(t.predictor, t._skill, color=[COL.get(g, "k") for g in t["[translated]"]])
    ax.axvline(0, color="k", lw=0.8); ax.invert_yaxis()
    ax.set_title(SPLIT_LABEL[s]); ax.set_xlabel("skill score"); ax.text(0.02, -0.25, note, transform=ax.transAxes, fontsize=6)
fig.tight_layout(); save(fig, "C_skill_score")

# ---- [translated] D：T_c [translated] ----
ce = RAW["crossing_errors_temp_extrap"]
ce = ce[~ce.predictor.str.startswith(FIXTURE_PREFIXES) & ce.reference_Tc_K.map(lambda x: not is_marker(x))]
pairs = sorted(ce.pair.unique()); preds = sorted(ce.predictor.unique(), key=lambda p: (GROUP_ORDER.index(pinfo(p)[1]), pinfo(p)[2]))
fig, ax = plt.subplots(figsize=(1.2 + 1.1 * len(pairs), 3.0))
w = 0.8 / max(len(preds), 1)
for j, p in enumerate(preds):
    for i, pair in enumerate(pairs):
        r = ce[(ce.predictor == p) & (ce.pair == pair)]
        if r.empty: continue
        r = r.iloc[0]; x = i + (j - len(preds) / 2) * w + w / 2
        if num(r.missed_crossings) > 0: ax.plot(x, 0, "x", color=COL.get(pinfo(p)[1], "k"), ms=6)
        elif not np.isnan(num(r.Tc_error_K)): ax.bar(x, num(r.Tc_error_K), w, color=COL.get(pinfo(p)[1], "k"))
    ax.bar(0, 0, 0, color=COL.get(pinfo(p)[1], "k"), label=pinfo(p)[0])
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(pairs))); ax.set_xticklabels([p.replace("_minus_", "\n− ") for p in pairs], fontsize=6)
ax.set_ylabel("T_c [translated] (K)"); ax.set_title("[translated]：[translated]phase T_c [translated]（x = [translated]crossing）")
ax.legend(fontsize=6, frameon=False, bbox_to_anchor=(1, 1))
fig.tight_layout(); save(fig, "D_tc_errors")

## export

- `tables_for_doc.xlsx`：[translated]（[translated]2a … [translated]6）[translated] Excel / [translated]table[translated]，[translated]table[translated]
- `md/*.md`：[translated] Markdown table[translated]
- `checks.txt`：[translated]check[translated]**[translated]table[translated]**
- `manifest.json`：data commit [translated] CSV [translated] SHA-256，[translated]

In [ ]:
out = pathlib.Path(OUT_DIR); (out / "md").mkdir(parents=True, exist_ok=True)
SHEETS = {"[translated]2a": T2A, "[translated]2b": T2B, "[translated]3": T3, "[translated]4": T4,
          "[translated]5a": T5["temp_extrap"][0], "[translated]5b": T5["phase_lopo"][0], "[translated]5c": T5["system_loso"][0], "[translated]6": T6}
NOTES = {"[translated]5a": T5["temp_extrap"][1], "[translated]5b": T5["phase_lopo"][1], "[translated]5c": T5["system_loso"][1]}

def to_md(df):
    df = df.astype(str)
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(r) + " |" for r in df.values)

with pd.ExcelWriter(out / "tables_for_doc.xlsx") as xw:
    for name, df in SHEETS.items():
        df = df.drop(columns=[c for c in df.columns if c.startswith("_")])
        df.to_excel(xw, sheet_name=name, index=False)
        text = f"# {name}\n\n" + (f"{NOTES[name]}\n\n" if name in NOTES else "") + to_md(df) + "\n"
        (out / "md" / f"{name}.md").write_text(text, encoding="utf-8")

(out / "checks.txt").write_text("\n".join(CHECKS) + "\n", encoding="utf-8")
manifest = {"commit": COMMIT, "source": SOURCE,
            "inputs": {p.name: sha256(p) for p in sorted(TABLES.glob("*.csv"))},
            "warnings": sum(c.startswith("⚠") for c in CHECKS)}
(out / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

shutil.make_archive(OUT_DIR, "zip", out)
print("written:", sorted(str(p.relative_to(out)) for p in out.rglob("*") if p.is_file()))
try:
    from google.colab import files; files.download(f"{OUT_DIR}.zip")
except ImportError:
    print(f"not in Colab; archive at {OUT_DIR}.zip")